# SpectraLite — Research Lab (`works.ipynb`)

Open via **File → Open notebook → GitHub** (do not double-click in Files).

## Rule

Each **PHASE N RUN** cell is **self-contained**:
sync git → deps → load model → run phase → write `results/`.

You can run **only that phase**. Use `FORCE_RERUN_PHASEN = True` to remeasure.

| Phase | In git? | Headline |
|-------|---------|----------|
| 0 | Yes | OPT-125M loads on A100; 73 Linears |
| 1 | Yes | Dense ~7.2ms prefill, ~7.1ms/tok; C4 PPL~28.7 (set FORCE_RERUN_PHASE1 for WT2/PTB) |
| 2 | Yes | Vanilla SVD: PPL explodes; latency **slower** than dense |
| 3 | **Runnable now** | Activation-aware SVD (whitening) |
| 4+ | Not yet | SpectraLite novelty next |


---
# Phase 0 — Environment & Smoke Test


In [ ]:
import sys, subprocess
from pathlib import Path

ROOT = Path("/content/Execution")
PKG = ROOT / "SpectraLite"
REPO = "https://github.com/PrabinDevkota/Execution.git"

if not (PKG / "spectralite").is_dir():
    subprocess.check_call(["git", "clone", REPO, str(ROOT)])
else:
    subprocess.check_call(["git", "-C", str(ROOT), "fetch", "origin"])
    subprocess.check_call(["git", "-C", str(ROOT), "reset", "--hard", "origin/main"])

if str(PKG) not in sys.path:
    sys.path.insert(0, str(PKG))

for _name in list(sys.modules):
    if _name == "spectralite" or _name.startswith("spectralite."):
        del sys.modules[_name]

print("Synced:", PKG)

FORCE_RERUN_PHASE0 = False

from spectralite.phase_runner import bootstrap_phase
from spectralite.artifacts import (
    is_phase_complete, save_phase0_artifacts, print_git_save_instructions, print_progress_dashboard,
)
from spectralite.system import print_environment_report
from spectralite.model_loader import print_model_load_summary, generate_text
from spectralite.model_analysis import print_full_model_analysis
from spectralite import gpu
from spectralite.utils import print_checklist, print_section

boot = bootstrap_phase(
    hard_reset=False, install_deps=True, require_cuda=True,
    model=globals().get("model"), tokenizer=globals().get("tokenizer"),
)
cfg, model, tokenizer = boot["cfg"], boot["model"], boot["tokenizer"]

if is_phase_complete("0", cfg) and not FORCE_RERUN_PHASE0:
    print_section("Phase 0 cached in results/ — skip (set FORCE_RERUN_PHASE0=True to redo)")
    print_progress_dashboard(cfg)
else:
    env_info = print_environment_report()
    mem_before = gpu.snapshot(label="Memory before loading")
    load_summary = print_model_load_summary(model, model_name=cfg.model_name)
    mem_after_load = gpu.snapshot(label="Memory after loading")
    analysis = print_full_model_analysis(model, model_name=cfg.model_name)
    generated = generate_text(model, tokenizer, cfg.smoke_prompt, max_new_tokens=cfg.max_new_tokens)
    print_section("Test Inference"); print(generated)
    mem_after_infer = gpu.snapshot(label="Memory after inference")
    gpu.print_memory_timeline([mem_before, mem_after_load, mem_after_infer])
    inference_ok = bool(generated and generated.strip())
    ok = inference_ok and gpu.is_cuda_available()
    print_checklist([
        ("GPU Ready", gpu.is_cuda_available()),
        ("Model Loaded", model is not None),
        ("Inference OK", inference_ok),
        ("Phase 0 Complete", ok),
    ])
    assert ok
    save_phase0_artifacts(env_info=env_info, load_summary=load_summary, analysis=analysis,
                         inference_ok=inference_ok, config=cfg)
    print_git_save_instructions()


---
# Phase 1 — Dense Baseline

Set `FORCE_RERUN_PHASE1 = True` once to refresh WT2/PTB PPL (first run had NaNs; code is fixed).


In [ ]:
import sys, subprocess
from pathlib import Path

ROOT = Path("/content/Execution")
PKG = ROOT / "SpectraLite"
REPO = "https://github.com/PrabinDevkota/Execution.git"

if not (PKG / "spectralite").is_dir():
    subprocess.check_call(["git", "clone", REPO, str(ROOT)])
else:
    subprocess.check_call(["git", "-C", str(ROOT), "fetch", "origin"])
    subprocess.check_call(["git", "-C", str(ROOT), "reset", "--hard", "origin/main"])

if str(PKG) not in sys.path:
    sys.path.insert(0, str(PKG))

for _name in list(sys.modules):
    if _name == "spectralite" or _name.startswith("spectralite."):
        del sys.modules[_name]

print("Synced:", PKG)

FORCE_RERUN_PHASE1 = False  # True => remeasure dense baseline with fixed PPL

from pathlib import Path
from spectralite.phase_runner import bootstrap_phase
from spectralite.benchmark import run_phase1_dense_baseline
from spectralite.artifacts import (
    is_phase_complete, read_json, print_git_save_instructions, print_progress_dashboard,
)
from spectralite.results_io import load_results
from spectralite.utils import print_checklist, print_section

boot = bootstrap_phase(
    hard_reset=False, install_deps=True, require_cuda=True,
    model=globals().get("model"), tokenizer=globals().get("tokenizer"),
)
cfg, model, tokenizer = boot["cfg"], boot["model"], boot["tokenizer"]

if is_phase_complete("1", cfg) and not FORCE_RERUN_PHASE1:
    print_section("Phase 1 cached — loading results/phase1_summary.json")
    row = read_json("phase1_summary.json", cfg).get("row", {})
else:
    out = run_phase1_dense_baseline(
        model, tokenizer, config=cfg,
        prompt_len=getattr(cfg, "latency_prompt_len", 128),
        gen_tokens=getattr(cfg, "latency_gen_tokens", 64),
        latency_warmup=getattr(cfg, "latency_warmup", 10),
        latency_reps_prefill=getattr(cfg, "latency_reps_prefill", 50),
        latency_reps_decode=getattr(cfg, "latency_reps_decode", 30),
        ppl_seq_len=getattr(cfg, "ppl_seq_len", 512),
        ppl_max_tokens=getattr(cfg, "ppl_max_tokens", 50_000),
        run_calflops=True, run_ppl=True,
        csv_name="phase1_dense_baselines.csv",
        phase="1", method="dense", persist_artifacts=True,
    )
    row = out["row"]

ok = row.get("empirical_flops_fwd") is not None and row.get("prefill_ms_mean") is not None
print_checklist([
    ("FLOPs", row.get("empirical_flops_fwd") is not None),
    ("Prefill", row.get("prefill_ms_mean") is not None),
    ("Decode", row.get("decode_ms_per_token_mean") is not None),
    ("C4 PPL", row.get("ppl_c4") not in (None, "")),
    ("WT2 PPL", row.get("ppl_wikitext2") not in (None, "", "nan")),
    ("Phase 1 Complete", ok and is_phase_complete("1", cfg)),
])
print_section("Phase 1 Outcome")
for k in ["prefill_ms_mean","decode_ms_per_token_mean","throughput_tokens_per_s",
          "ppl_wikitext2","ppl_ptb","ppl_c4","empirical_flops_fwd"]:
    print(f"  {k}: {row.get(k)}")
print(load_results(Path(cfg.results_dir)/"phase1_dense_baselines.csv").tail(2).to_string(index=False))
print_progress_dashboard(cfg)
print_git_save_instructions()
assert ok


---
# Phase 2 — Vanilla Truncated SVD

Cached in git: r=0.5/0.4/0.3 → C4 PPL ~922/2953/6256 vs dense ~28.7;  
decode ~9.8 ms/tok vs dense ~7.1 (**no speedup**). Set `FORCE_RERUN_PHASE2 = True` to redo.


In [ ]:
import sys, subprocess
from pathlib import Path

ROOT = Path("/content/Execution")
PKG = ROOT / "SpectraLite"
REPO = "https://github.com/PrabinDevkota/Execution.git"

if not (PKG / "spectralite").is_dir():
    subprocess.check_call(["git", "clone", REPO, str(ROOT)])
else:
    subprocess.check_call(["git", "-C", str(ROOT), "fetch", "origin"])
    subprocess.check_call(["git", "-C", str(ROOT), "reset", "--hard", "origin/main"])

if str(PKG) not in sys.path:
    sys.path.insert(0, str(PKG))

for _name in list(sys.modules):
    if _name == "spectralite" or _name.startswith("spectralite."):
        del sys.modules[_name]

print("Synced:", PKG)

FORCE_RERUN_PHASE2 = False

from spectralite.phase_runner import bootstrap_phase
from spectralite.phase2 import run_phase2_vanilla_svd_sweep
from spectralite.artifacts import (
    is_phase_complete, read_json, print_git_save_instructions, print_progress_dashboard,
)
from spectralite.utils import print_checklist, print_section

boot = bootstrap_phase(
    hard_reset=False, install_deps=True, require_cuda=True,
    model=globals().get("model"), tokenizer=globals().get("tokenizer"),
)
cfg, model, tokenizer = boot["cfg"], boot["model"], boot["tokenizer"]

if is_phase_complete("2", cfg) and not FORCE_RERUN_PHASE2:
    print_section("Phase 2 cached — loading results/phase2_summary.json")
    phase2 = read_json("phase2_summary.json", cfg)
else:
    phase2 = run_phase2_vanilla_svd_sweep(
        model, tokenizer, config=cfg,
        rank_ratios=(0.5, 0.4, 0.3),
        ppl_seq_len=getattr(cfg, "ppl_seq_len", 512),
        ppl_max_tokens=30_000,
        latency_reps_prefill=30, latency_reps_decode=20,
        csv_name="phase2_vanilla_svd.csv",
    )

rows = phase2.get("rows", [])
ok = is_phase_complete("2", cfg) and len(rows) > 0
print_checklist([
    ("Artifacts", is_phase_complete("2", cfg)),
    ("Rows", len(rows) > 0),
    ("Phase 2 Complete", ok),
])
print_section("Phase 2 Outcome")
for row in rows:
    print(f"  {row.get('method')}: prefill={row.get('prefill_ms_mean')} decode={row.get('decode_ms_per_token_mean')} C4={row.get('ppl_c4')}")
print_progress_dashboard(cfg)
print_git_save_instructions()
assert ok


---
# Phase 3 — Activation-Aware SVD (ASVD / SVD-LLM style)

**Self-contained RUN cell** below.

**What it does (plan Phase 3):**
1. WikiText-2 calibration forwards
2. Hook Linear inputs → covariance ``C = XᵀX/n + λI``
3. Cholesky whitening → truncated SVD on ``W L`` → fuse factors → ``LowRankLinear``
4. Same rank ratios as Phase 2 (`0.5, 0.4, 0.3`) for fair comparison
5. Measure PPL / latency / FLOPs; write `results/phase3_*`

**Compare to Phase 2:** whitening should improve PPL at matched rank ratios (still may not speed up — latency gate is Phase 6).

Set `FORCE_RERUN_PHASE3 = True` to remeasure.


In [ ]:
import sys, subprocess
from pathlib import Path

ROOT = Path("/content/Execution")
PKG = ROOT / "SpectraLite"
REPO = "https://github.com/PrabinDevkota/Execution.git"

if not (PKG / "spectralite").is_dir():
    subprocess.check_call(["git", "clone", REPO, str(ROOT)])
else:
    subprocess.check_call(["git", "-C", str(ROOT), "fetch", "origin"])
    subprocess.check_call(["git", "-C", str(ROOT), "reset", "--hard", "origin/main"])

if str(PKG) not in sys.path:
    sys.path.insert(0, str(PKG))

for _name in list(sys.modules):
    if _name == "spectralite" or _name.startswith("spectralite."):
        del sys.modules[_name]

print("Synced:", PKG)

FORCE_RERUN_PHASE3 = False

from spectralite.phase_runner import bootstrap_phase
from spectralite.phase3 import run_phase3_activation_aware_sweep
from spectralite.artifacts import (
    is_phase_complete, read_json, print_git_save_instructions, print_progress_dashboard,
)
from spectralite.utils import print_checklist, print_section

boot = bootstrap_phase(
    hard_reset=False, install_deps=True, require_cuda=True,
    model=globals().get("model"), tokenizer=globals().get("tokenizer"),
)
cfg, model, tokenizer = boot["cfg"], boot["model"], boot["tokenizer"]

if is_phase_complete("3", cfg) and not FORCE_RERUN_PHASE3:
    print_section("Phase 3 cached — loading results/phase3_summary.json")
    phase3 = read_json("phase3_summary.json", cfg)
else:
    phase3 = run_phase3_activation_aware_sweep(
        model,
        tokenizer,
        config=cfg,
        rank_ratios=(0.5, 0.4, 0.3),
        calib_num_sequences=getattr(cfg, "calib_num_sequences", 32),
        calib_seq_len=min(getattr(cfg, "calib_seq_len", 2048), 512),  # Colab-speed default
        calib_batch_size=2,
        ridge=1e-2,
        ppl_seq_len=getattr(cfg, "ppl_seq_len", 512),
        ppl_max_tokens=30_000,
        latency_reps_prefill=30,
        latency_reps_decode=20,
        csv_name="phase3_activation_aware_svd.csv",
    )

rows = phase3.get("rows", [])
ok = is_phase_complete("3", cfg) and len(rows) > 0
print_checklist([
    ("Artifacts", is_phase_complete("3", cfg)),
    ("Rows", len(rows) > 0),
    ("Phase 3 Complete", ok),
])
print_section("Phase 3 Outcome (vs remember Phase 2 C4 ~922/2953/6256)")
for row in rows:
    print(
        f"  {row.get('method')}: prefill={row.get('prefill_ms_mean')} "
        f"decode={row.get('decode_ms_per_token_mean')} "
        f"C4={row.get('ppl_c4')} WT2={row.get('ppl_wikitext2')}"
    )
print_progress_dashboard(cfg)
print_git_save_instructions()
assert ok


---
# Phases 4–8 (later)

Each will be one self-contained RUN cell when implemented.


In [ ]:
print("Phases 4–8 not implemented yet.")

---
# Save results

Download `SpectraLite/results/` → PC → tell Cursor **uploaded**.
